In [ ]:
def read_file_as_df(file_name):
    import pandas as pd
    import csv

    import sys
    import pandas as pd

    maxInt = sys.maxsize

    while True:
        # decrease the maxInt value by factor 10
        # as long as the OverflowError occurs.

        try:
            csv.field_size_limit(maxInt)
            break
        except OverflowError:
            maxInt = int(maxInt/10)

    file = []
    col = []

    with open(file_name, encoding='latin-1') as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=';')
        line_count = 0
        for row in csv_reader:
            if line_count==0:
                for r in row:
                    col.append(r)
                line_count+=1
            else:
                line = []
                for r in row:
                    line.append(r)
                file.append(line)
                line_count += 1


    df = pd.DataFrame(file, columns = col)
    return df

In [ ]:
import nltk

nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('portuguese')

import string
import pandas as pd

[nltk_data] Downloading package stopwords to /home/alice/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
def remove_stopwords(text):
    all_words = text.split(" ")
    clean_text = [i for i in all_words if i not in stopwords and i!=""]
    return " ".join(clean_text)

def clean_text(x):
  return " ".join(str(x.translate(str.maketrans('', '', string.punctuation))).split()).lower()

def get_age_class(age):
    if age == 'a17-30': return 0
    elif age == 'a31-42': return 1
    else: return 2

In [ ]:
df = read_file_as_df('../data/e-SIC1BR.csv')

In [ ]:
df.head()

,age-bracket,gender,education,profession,req-text,resp-text,resp-type,clarity,service,1funct-request,...,55work-response,56achieve-response,57leisure-response,58home-response,59money-response,60relig-response,61death-response,62assent-response,63nonfl-response,64filler-response
0,a31-42,m,e2,academy,"Prezados, Gostaria de solicitar informações so...","Não dispomos de vaga em aberto, para o cargo d...",Acesso Concedido,5,5,"0,4531",...,"0,0612","0,102","0,0204",0,0,0,0,0,0,0
1,a31-42,m,e2,academy,"Prezados, Gostaria de solicitar informações so...","Prezado cidadão, O quadro de referência de ser...",Acesso Concedido,5,5,"0,4531",...,"0,0394","0,0472","0,0157",0,"0,0079",0,0,0,"0,0079",0
2,a31-42,m,e2,academy,"Prezados, Gostaria de solicitar informações so...","Sr. JOMAR BORGES DOS SANTOS, boa tarde. Em res...",Acesso Concedido,5,5,"0,4531",...,"0,0541","0,0541",0,0,0,"0,027",0,0,"0,027",0
3,a31-42,m,e2,academy,"Prezados, Gostaria de solicitar informações so...","Prezado Jomar, Encaminho resposta à sua solici...",Acesso Concedido,5,5,"0,4531",...,"0,0549","0,0706","0,0196",0,"0,0275","0,0078",0,0,"0,0235",0
4,a43-xx,na,na,na,gostaria de saber se a empresa a qual min refi...,"Prezado(a) Senhor(a), Informamos que a Secreta...",Acesso Concedido,1,1,"0,4286",...,"0,0476","0,0476",0,0,"0,0476","0,0159",0,0,"0,0476",0


In [ ]:
df = read_file_as_df('../data/e-SIC1BR.csv')
df = df[['age-bracket', 'req-text']]

df["req-text-clean"] = df['req-text'].apply(lambda x: clean_text(x))
df["req-text-clean"] = df['req-text-clean'].apply(lambda x: remove_stopwords(x))
df["Age"] = df['age-bracket'].apply(lambda x: get_age_class(x))

X = df["req-text-clean"]
y = df['Age']

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=115, stratify=y)
#df_train, df_test = train_test_split(df, test_size=0.2, random_state=289, stratify=y)


In [ ]:
train_df.head()

,age-bracket,req-text,req-text-clean,Age
3415,a43-xx,Gostaria de adotar um casal de cacatuas de um ...,gostaria adotar casal cacatuas criador argenti...,2
24254,a43-xx,"Minha esposa ganhou neném, mas na situação de ...",esposa ganhou neném situação natmorto nasceu m...,2
269,a43-xx,Peço a informação referente ao nome de todos o...,peço informação referente nome todos candidato...,2
24435,a17-30,o quantitativo de cargos ocupados e vagos do c...,quantitativo cargos ocupados vagos cargo técni...,0
15926,a17-30,Sou Aluna da Academia de Policia Militar Costa...,aluna academia policia militar costa verde pmm...,0


In [ ]:
train_df[train_df['req-text'].isna()]

,age-bracket,req-text,req-text-clean,Age


In [ ]:
train_df[train_df['req-text']=='nan']

,age-bracket,req-text,req-text-clean,Age
44233,a31-42,nan,nan,1
42511,a31-42,nan,nan,1


In [ ]:
train_df = train_df[train_df['req-text']!='nan']

In [ ]:
train_df.to_csv('../data/train.csv', index=True)
test_df.to_csv('../data/test.csv', index=True)

In [ ]:
unique_words = list(set(" ".join(train_df["req-text-clean"].to_list()).split()))
print(len(unique_words))
threshold = train_df.shape[0]*0.01
word_counts = train_df["req-text-clean"].str.split().explode().value_counts()
row_counts = pd.DataFrame({'word': word_counts.index, 'count': word_counts.values})
row_counts['count'] = row_counts['word'].apply(lambda word: train_df["req-text-clean"].str.contains(word, regex=False).sum())
selected_words = row_counts[row_counts['count']>threshold]['word'].to_list()

85317


In [ ]:
train_df.shape[0]

38207

In [ ]:
len(selected_words)

2769

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

text = train_df["req-text-clean"].to_list()

vectorizer = TfidfVectorizer(vocabulary=selected_words)
train_matrix = vectorizer.fit_transform(text)
train_matrix = train_matrix.toarray()

tf_idf = pd.DataFrame(data=train_matrix, columns=selected_words)

X = tf_idf
y =  train_df['Age']

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 100],  # Regularization parameter
    'max_iter': [1000, 10000, -1] # Maximum number of iterations
}

svc = SVC(kernel='linear', decision_function_shape='ovr')
grid_search = GridSearchCV(estimator=svc, param_grid=param_grid, cv=5, scoring='f1_macro', verbose=2)

# Fit the model to the training data
grid_search.fit(X, y)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


/home/alice/.local/lib/python3.9/site-packages/sklearn/svm/_base.py:301: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[CV] END ..................C=0.1, gamma=scale, max_iter=1000; total time= 5.5min


/home/alice/.local/lib/python3.9/site-packages/sklearn/svm/_base.py:301: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[CV] END ..................C=0.1, gamma=scale, max_iter=1000; total time= 5.4min


...

[CV] END ..................C=100, gamma=auto, max_iter=10000; total time=45.1min


/home/alice/.local/lib/python3.9/site-packages/sklearn/svm/_base.py:301: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[CV] END ..................C=100, gamma=auto, max_iter=10000; total time=28.8min
[CV] END ....................C=100, gamma=auto, max_iter=-1; total time=391.2min
[CV] END ....................C=100, gamma=auto, max_iter=-1; total time=397.6min
[CV] END ....................C=100, gamma=auto, max_iter=-1; total time=351.9min
[CV] END ....................C=100, gamma=auto, max_iter=-1; total time=347.0min
[CV] END ....................C=100, gamma=auto, max_iter=-1; total time=343.7min


/home/alice/.local/lib/python3.9/site-packages/sklearn/svm/_base.py:301: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


GridSearchCV(cv=5, estimator=SVC(kernel='linear'),
             param_grid={'C': [0.1, 1, 100], 'gamma': ['scale', 'auto'],
                         'max_iter': [1000, 10000, -1]},
             scoring='accuracy', verbose=2)

In [ ]:
grid_search.best_estimator_, grid_search.best_score_

model = grid_search.best_estimator_

In [ ]:
from sklearn.svm import SVC

svc = SVC(kernel='linear', decision_function_shape='ovr')

svc.fit(X, y)

SVC(kernel='linear')

In [ ]:
tf_idf.columns

Index(['gostaria', 'informações', 'solicito', 'saber', 'sobre', 'informação',
       'federal', 'nº', 'dados', 'acesso',
       ...
       '082', '092', 'func', 'outo', 'cnicos', 'ot', 'cadastr', 'argos',
       'desen', 'volvimento'],
      dtype='object', length=2769)

In [ ]:
len(model.coef_[0])

2776

In [ ]:
df_coef = pd.DataFrame(data=svc.coef_, columns=tf_idf.columns)

In [ ]:
df_coef.head()

,gostaria,informações,solicito,saber,sobre,informação,federal,nº,dados,acesso,...,082,092,func,outo,cnicos,ot,cadastr,argos,desen,volvimento
0,2.050285,-0.017188,-1.617415,-0.778614,0.032923,-0.962688,-0.407322,0.757566,1.039780,1.470564,...,0.00000,-0.326543,0.000000,0.792601,0.222505,0.000000,0.0,0.000000,0.000000,0.000000
1,4.015860,0.368125,0.861079,-0.666733,-0.658767,-0.530750,-0.119015,0.618261,1.404853,1.302993,...,0.00000,0.000000,-0.448612,0.792601,0.222505,-0.455444,0.0,0.186976,-0.429931,-0.429931
2,3.854217,0.608594,3.023504,-0.600474,-0.603698,0.414455,0.133163,0.177975,0.858657,0.442995,...,-0.24107,0.326543,-0.352289,0.000000,0.000000,-0.455444,0.0,0.000000,-0.429931,-0.429931


In [ ]:
df_coef = df_coef.T

In [ ]:
df_coef.head()

,0,1,2
gostaria,2.050285,4.015860,3.854217
informações,-0.017188,0.368125,0.608594
solicito,-1.617415,0.861079,3.023504
saber,-0.778614,-0.666733,-0.600474
sobre,0.032923,-0.658767,-0.603698


In [ ]:
def select_abs(col, perc): # mesma quantidade de palavras positivas e negativas
  n = int(len(df_coef) * perc)

  df_coef_1 = df_coef.copy()
  df_coef_1 = df_coef_1[[col]]

  df_coef_1['abs_values'] = df_coef_1[col].abs()

  sorted_df = df_coef_1.sort_values(by='abs_values', ascending=False)

  return sorted_df.head(n)

coef1 = select_abs(0, 0.25)
coef2 = select_abs(1, 0.25)
coef3 = select_abs(2, 0.25)

In [ ]:
def select_(col, perc): # mesma quantidade de palavras positivas e negativas
  df_coef_1 = df_coef.copy()
  df_coef_1 = df_coef_1[[col]]

  str_col = str(col)

  df_coef_1.rename(columns={col: str_col}, inplace=True)

  df_coef_1_pos = df_coef_1[df_coef_1[str_col]>0]
  df_coef_1_neg= df_coef_1[df_coef_1[str_col]<0]
  df_coef_1_zero = df_coef_1[df_coef_1[str_col]==0]

  df_coef_1_pos = df_coef_1_pos.sort_values(by=str_col, ascending=True)
  df_coef_1_neg = df_coef_1_neg.sort_values(by=str_col, key=lambda x: abs(x))

  pos_perc = df_coef_1_pos.head(int(len(df_coef_1_pos) * perc))
  neg_perc = df_coef_1_neg.head(int(len(df_coef_1_neg) * perc))

  indices_to_drop = pd.concat([pos_perc, neg_perc, df_coef_1_zero]).index

  df_filtered = df_coef_1.drop(indices_to_drop)

  return df_filtered.sort_values(by=str_col, ascending=False)

coef1 = select_(0, 0.5)
coef2 = select_(1, 0.5)
coef3 = select_(2, 0.5)

In [ ]:
coef1.head(10), coef2.head(10), coef3.head(10)

(                  0
 estudante  3.067908
 políticas  2.493509
 requer     2.485529
 005        2.427476
 ola        2.384320
 planeja    2.295623
 125272011  2.253966
 logo       2.230794
 funciona   2.227420
 feita      2.223379,
                         1
 gostaria         4.015860
 olá              3.193610
 requisito        3.099176
 políticas        2.988923
 prezadoa         2.731642
 sim              2.605873
 minas            2.298892
 exista           2.269040
 disponibilizada  2.247642
 questões         2.245325,
                    2
 gostaria    3.854217
 entro       3.335918
 solicito    3.023504
 sere        2.922476
 prezadoa    2.744929
 venho       2.734334
 boa         2.632102
 realizando  2.621025
 acadêmica   2.562396
 aplicação   2.443135)

In [ ]:
coef1['0']['estudante'], coef2['1']['senhores'], coef3['2']['senhores']

(3.0679075460247596, -4.495259336530953, -2.6158072366238514)

In [ ]:
train_df['glex1']=0
train_df['glex2']=0
train_df['glex3']=0

In [ ]:
train_df.reset_index(inplace=True)

In [ ]:
# dict 1
for index, row in train_df.iterrows():
    sum = 0
    text = row["req-text-clean"].split()
    for w in text:
      if w in coef1.index.tolist():
        sum += coef1['0'][w]*tf_idf[w][index]
    sum = sum + svc.intercept_[0]
    train_df[f'glex1'][index] = sum

/tmp/ipykernel_28074/2453714398.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f'glex1'][index] = sum


In [ ]:
# dict 2
for index, row in train_df.iterrows():
    sum = 0
    text = row["req-text-clean"].split()
    for w in text:
      if w in coef2.index.tolist():
        sum += coef2['1'][w]*tf_idf[w][index]
    sum = sum + svc.intercept_[1]
    train_df[f'glex2'][index] = sum

/tmp/ipykernel_28074/1943462446.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f'glex2'][index] = sum


In [ ]:
# dict 3
for index, row in train_df.iterrows():
    sum = 0
    text = row["req-text-clean"].split()
    for w in text:
      if w in coef3.index.tolist():
        sum += coef3['2'][w]*tf_idf[w][index]
    sum = sum + svc.intercept_[2]
    train_df[f'glex3'][index] = sum

/tmp/ipykernel_28074/1633217855.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f'glex3'][index] = sum


In [ ]:
train_df.groupby(['Age'])[['glex1', 'glex2', 'glex3']].describe().T

Age                     0             1             2
glex1 count  13415.000000  13580.000000  11212.000000
      mean       0.810134     -0.637688     -0.341426
      std        1.797505      2.062626      1.701676
      min      -25.277380    -27.319891    -24.899050
      25%       -0.058778     -1.286626     -1.173453
      50%        0.789039     -0.562915     -0.323090
      75%        1.513151      0.351010      0.534609
      max       44.315433     36.619076     17.711731
glex2 count  13415.000000  13580.000000  11212.000000
      mean       1.314079      0.495896     -0.582183
      std        2.031765      2.084753      1.813236
      min      -11.324384    -18.062178    -17.847099
      25%        0.287210     -0.568456     -1.404108
      50%        1.156413      0.417359     -0.663605
      75%        2.066302      1.525907      0.308106
      max       62.991473     28.187881     14.730723
glex3 count  13415.000000  13580.000000  11212.000000
      mean       0.839614      1.270065     -0.413349
      std        1.796572      2.280684      1.711277
      min      -13.989171    -22.773748    -18.050049
      25%       -0.083759      0.111468     -1.166309
      50%        0.731511      0.988952     -0.435510
      75%        1.602213      1.936201      0.444807
      max       45.778331     36.661035     19.997314

In [ ]:
train_df['c'] = -1

for index, row in train_df.iterrows():
  if row['glex1'] > 0 and row['glex2'] > 0 and row['glex3'] > 0:
    train_df['c'][index] = 0
  elif row['glex1'] < 0 and row['glex2'] > 0 and row['glex3'] > 0:
    train_df['c'][index] = 1
  elif row['glex1'] < 0 and row['glex2'] < 0 and row['glex3'] < 0:
    train_df['c'][index] = 2

/tmp/ipykernel_28074/4284971904.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['c'][index] = 1
/tmp/ipykernel_28074/4284971904.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['c'][index] = 0
/tmp/ipykernel_28074/4284971904.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['c'][index] = 2


In [ ]:
print(f'Instâncias totais: {train_df.shape[0]}')
print(f'Instâncias classificadas: {train_df[train_df.c!=-1].shape[0]}')
print(f'Porcentagem classificadas: {train_df[train_df.c!=-1].shape[0]/train_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {train_df[(train_df.c!=-1) & (train_df.c==train_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {train_df[(train_df.c!=-1) & (train_df.c==train_df.Age)].shape[0]/train_df[train_df.c!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {train_df[(train_df.c!=-1) & (train_df.c==train_df.Age)].shape[0]/train_df.shape[0]}')

# as palavras não esto conseguindo separar bem (1% muito, restringir menos)

Instâncias totais: 38207
Instâncias classificadas: 25742
Porcentagem classificadas: 0.6737508833459838
Instâncias classificadas corretamente: 15449
Porcentagem classificadas corretamente: 0.6001476186776474
Porcentagem classificadas corretamente das totais: 0.40434998822205354


In [ ]:
test_matrix = vectorizer.transform(test_df["req-text-clean"].to_list())
test_matrix = test_matrix.toarray()

tf_idf_test = pd.DataFrame(data=test_matrix, columns=selected_words)

#test_df.reset_index(inplace=True)

test_df['glex1']=0
test_df['glex2']=0
test_df['glex3']=0

# dict 1
for index, row in test_df.iterrows():
    sum = 0
    text = row["req-text-clean"].split()
    for w in text:
      if w in coef1.index.tolist():
        sum += coef1['0'][w]*tf_idf_test[w][index]
    sum = sum + svc.intercept_[0]
    test_df[f'glex1'][index] = sum

# dict 1
for index, row in test_df.iterrows():
    sum = 0
    text = row["req-text-clean"].split()
    for w in text:
      if w in coef2.index.tolist():
        sum += coef2['1'][w]*tf_idf_test[w][index]
    sum = sum + svc.intercept_[1]
    test_df[f'glex2'][index] = sum

# dict 1
for index, row in test_df.iterrows():
    sum = 0
    text = row["req-text-clean"].split()
    for w in text:
      if w in coef3.index.tolist():
        sum += coef3['2'][w]*tf_idf_test[w][index]
    sum = sum + svc.intercept_[2]
    test_df[f'glex3'][index] = sum

test_df.groupby(['Age'])[['glex1', 'glex2', 'glex3']].describe().T

/tmp/ipykernel_28074/739443472.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f'glex1'][index] = sum
/tmp/ipykernel_28074/739443472.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f'glex2'][index] = sum
/tmp/ipykernel_28074/739443472.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f'glex3'][index] = sum


Age                    0            1            2
glex1 count  3354.000000  3396.000000  2803.000000
      mean      0.667189    -0.464050    -0.296721
      std       1.783932     1.918013     1.601261
      min     -23.386505   -14.101295   -13.252122
      25%      -0.231297    -1.151318    -1.096449
      50%       0.641397    -0.396685    -0.263134
      75%       1.429570     0.489178     0.546146
      max      24.591079    19.749237     8.110370
glex2 count  3354.000000  3396.000000  2803.000000
      mean      1.100525     0.447468    -0.550269
      std       1.994080     1.936367     1.719560
      min     -10.637223   -13.944603   -17.819542
      25%       0.016307    -0.599812    -1.342009
      50%       0.978123     0.372074    -0.547117
      75%       1.857137     1.397052     0.307171
      max      36.780358    22.548237     9.081882
glex3 count  3354.000000  3396.000000  2803.000000
      mean      0.730928     0.954824    -0.405418
      std       1.839025     2.047769     1.793467
      min     -10.972681   -17.183546   -19.930740
      25%      -0.149243    -0.136941    -1.163165
      50%       0.607313     0.764337    -0.401218
      75%       1.459009     1.730518     0.452427
      max      45.948042    33.620934    22.427292

In [ ]:
test_df['c'] = -1

for index, row in test_df.iterrows():
  if row['glex1'] > 0 and row['glex2'] > 0 and row['glex3'] > 0: # delimitaria só pelo glex da regra
    test_df['c'][index] = 0
  elif row['glex1'] < 0 and row['glex2'] > 0 and row['glex3'] > 0:
    test_df['c'][index] = 1
  elif row['glex1'] < 0 and row['glex2'] < 0 and row['glex3'] < 0:
    test_df['c'][index] = 2

/tmp/ipykernel_28074/4243654698.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c'][index] = 2
/tmp/ipykernel_28074/4243654698.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c'][index] = 1
/tmp/ipykernel_28074/4243654698.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c'][index] = 0


In [ ]:
print(f'Instâncias totais: {test_df.shape[0]}')
print(f'Instâncias classificadas: {test_df[test_df.c!=-1].shape[0]}')
print(f'Porcentagem classificadas: {test_df[test_df.c!=-1].shape[0]/test_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {test_df[(test_df.c!=-1) & (test_df.c==test_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {test_df[(test_df.c!=-1) & (test_df.c==test_df.Age)].shape[0]/test_df[test_df.c!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {test_df[(test_df.c!=-1) & (test_df.c==test_df.Age)].shape[0]/test_df.shape[0]}')

Instâncias totais: 9553
Instâncias classificadas: 6389
Porcentagem classificadas: 0.6687951428870512
Instâncias classificadas corretamente: 3603
Porcentagem classificadas corretamente: 0.563938018469244
Porcentagem classificadas corretamente das totais: 0.37715900764157856


In [ ]:
df_stat = train_df.groupby(['Age'])[['glex1', 'glex2', 'glex3']].describe().T

In [ ]:
test_df['c50'] = -1

for index, row in test_df.iterrows():
  if row['glex1'] > df_stat[0]['glex1']['50%'] and row['glex2'] > 0 and row['glex3'] > 0: # delimitaria só pelo glex da regra
    test_df['c50'][index] = 0
  elif row['glex1'] < 0 and row['glex2'] > df_stat[1]['glex2']['50%'] and row['glex3'] > 0:
    test_df['c50'][index] = 1
  elif row['glex1'] < 0 and row['glex2'] < 0 and row['glex3'] < df_stat[2]['glex3']['50%']:
    test_df['c50'][index] = 2

print(f'Instâncias totais: {test_df.shape[0]}')
print(f'Instâncias classificadas: {test_df[test_df.c50!=-1].shape[0]}')
print(f'Porcentagem classificadas: {test_df[test_df.c50!=-1].shape[0]/test_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {test_df[(test_df.c50!=-1) & (test_df.c50==test_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {test_df[(test_df.c50!=-1) & (test_df.c50==test_df.Age)].shape[0]/test_df[test_df.c50!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {test_df[(test_df.c50!=-1) & (test_df.c50==test_df.Age)].shape[0]/test_df.shape[0]}')

/tmp/ipykernel_28074/2428722408.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c50'][index] = 2
/tmp/ipykernel_28074/2428722408.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c50'][index] = 1
/tmp/ipykernel_28074/2428722408.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c50'][index] = 0


Instâncias totais: 9553
Instâncias classificadas: 4330
Porcentagem classificadas: 0.4532607557835235
Instâncias classificadas corretamente: 2683
Porcentagem classificadas corretamente: 0.6196304849884526
Porcentagem classificadas corretamente das totais: 0.28085418193237727


In [ ]:
test_df['c5025'] = -1

for index, row in test_df.iterrows():
  if row['glex1'] > df_stat[0]['glex1']['50%'] and row['glex2'] > df_stat[0]['glex2']['25%'] and row['glex3'] > df_stat[0]['glex3']['25%']: # delimitaria só pelo glex da regra
    test_df['c5025'][index] = 0
  elif row['glex1'] < df_stat[1]['glex1']['25%'] and row['glex2'] > df_stat[1]['glex2']['50%'] and row['glex3'] > df_stat[1]['glex3']['25%']:
    test_df['c5025'][index] = 1
  elif row['glex1'] < df_stat[2]['glex1']['25%'] and row['glex2'] < df_stat[2]['glex2']['25%'] and row['glex3'] < df_stat[2]['glex3']['50%']:
    test_df['c5025'][index] = 2

print(f'Instâncias totais: {test_df.shape[0]}')
print(f'Instâncias classificadas: {test_df[test_df.c5025!=-1].shape[0]}')
print(f'Porcentagem classificadas: {test_df[test_df.c5025!=-1].shape[0]/test_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {test_df[(test_df.c5025!=-1) & (test_df.c5025==test_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {test_df[(test_df.c5025!=-1) & (test_df.c5025==test_df.Age)].shape[0]/test_df[test_df.c5025!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {test_df[(test_df.c5025!=-1) & (test_df.c5025==test_df.Age)].shape[0]/test_df.shape[0]}')

/tmp/ipykernel_28074/3397982795.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c5025'][index] = 0
/tmp/ipykernel_28074/3397982795.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c5025'][index] = 2
/tmp/ipykernel_28074/3397982795.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c5025'][index] = 1


Instâncias totais: 9553
Instâncias classificadas: 2602
Porcentagem classificadas: 0.2723751701036324
Instâncias classificadas corretamente: 1688
Porcentagem classificadas corretamente: 0.6487317448116833
Porcentagem classificadas corretamente das totais: 0.17669841934470848


In [ ]:
test_df_class = test_df[test_df.c5025!=-1]

In [ ]:
test_df_class.shape

(2751, 13)

In [ ]:
test_df_class.to_csv('../data/dict-class.csv', encoding='latin-1')

In [ ]:
test_df['cmean'] = -1

for index, row in test_df.iterrows():
  if row['glex1'] > df_stat[0]['glex1']['mean'] and row['glex2'] > 0 and row['glex3'] > 0: # delimitaria só pelo glex da regra
    test_df['cmean'][index] = 0
  elif row['glex1'] < 0 and row['glex2'] > df_stat[1]['glex2']['mean'] and row['glex3'] > 0:
    test_df['cmean'][index] = 1
  elif row['glex1'] < 0 and row['glex2'] < 0 and row['glex3'] < df_stat[2]['glex3']['mean']:
    test_df['cmean'][index] = 2

print(f'Instâncias totais: {test_df.shape[0]}')
print(f'Instâncias classificadas: {test_df[test_df.cmean!=-1].shape[0]}')
print(f'Porcentagem classificadas: {test_df[test_df.cmean!=-1].shape[0]/test_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {test_df[(test_df.cmean!=-1) & (test_df.cmean==test_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {test_df[(test_df.cmean!=-1) & (test_df.cmean==test_df.Age)].shape[0]/test_df[test_df.cmean!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {test_df[(test_df.cmean!=-1) & (test_df.cmean==test_df.Age)].shape[0]/test_df.shape[0]}')

/tmp/ipykernel_28074/2209683470.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['cmean'][index] = 2
/tmp/ipykernel_28074/2209683470.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['cmean'][index] = 1
/tmp/ipykernel_28074/2209683470.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['cmean'][index] = 0


Instâncias totais: 9553
Instâncias classificadas: 4218
Porcentagem classificadas: 0.44153669004501206
Instâncias classificadas corretamente: 2631
Porcentagem classificadas corretamente: 0.6237553342816501
Porcentagem classificadas corretamente das totais: 0.2754108656966398


In [ ]:
test_df['c75'] = -1

for index, row in test_df.iterrows():
  if row['glex1'] > df_stat[0]['glex1']['75%'] and row['glex2'] > 0 and row['glex3'] > 0: # delimitaria só pelo glex da regra
    test_df['c75'][index] = 0
  elif row['glex1'] < 0 and row['glex2'] > df_stat[1]['glex2']['75%'] and row['glex3'] > 0:
    test_df['c75'][index] = 1
  elif row['glex1'] < 0 and row['glex2'] < 0 and row['glex3'] < df_stat[2]['glex3']['75%']:
    test_df['c75'][index] = 2

print(f'Instâncias totais: {test_df.shape[0]}')
print(f'Instâncias classificadas: {test_df[test_df.c75!=-1].shape[0]}')
print(f'Porcentagem classificadas: {test_df[test_df.c75!=-1].shape[0]/test_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {test_df[(test_df.c75!=-1) & (test_df.c75==test_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {test_df[(test_df.c75!=-1) & (test_df.c75==test_df.Age)].shape[0]/test_df[test_df.c75!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {test_df[(test_df.c75!=-1) & (test_df.c75==test_df.Age)].shape[0]/test_df.shape[0]}')

/tmp/ipykernel_28074/2537321040.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c75'][index] = 2
/tmp/ipykernel_28074/2537321040.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c75'][index] = 0
/tmp/ipykernel_28074/2537321040.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c75'][index] = 1


Instâncias totais: 9553
Instâncias classificadas: 3536
Porcentagem classificadas: 0.3701455040301476
Instâncias classificadas corretamente: 2075
Porcentagem classificadas corretamente: 0.5868212669683258
Porcentagem classificadas corretamente das totais: 0.21720925363760074


In [ ]:
test_df['c7525'] = -1

for index, row in test_df.iterrows():
  if row['glex1'] > df_stat[0]['glex1']['75%'] and row['glex2'] > df_stat[0]['glex2']['25%'] and row['glex3'] > df_stat[0]['glex3']['25%']: # delimitaria só pelo glex da regra
    test_df['c7525'][index] = 0
  elif row['glex1'] < df_stat[1]['glex1']['25%'] and row['glex2'] > df_stat[1]['glex2']['75%'] and row['glex3'] > df_stat[1]['glex3']['25%']:
    test_df['c7525'][index] = 1
  elif row['glex1'] < df_stat[2]['glex1']['25%'] and row['glex2'] < df_stat[2]['glex2']['25%'] and row['glex3'] < df_stat[2]['glex3']['75%']:
    test_df['c7525'][index] = 2

print(f'Instâncias totais: {test_df.shape[0]}')
print(f'Instâncias classificadas: {test_df[test_df.c7525!=-1].shape[0]}')
print(f'Porcentagem classificadas: {test_df[test_df.c7525!=-1].shape[0]/test_df.shape[0]}')
print(f'Instâncias classificadas corretamente: {test_df[(test_df.c7525!=-1) & (test_df.c7525==test_df.Age)].shape[0]}')
print(f'Porcentagem classificadas corretamente: {test_df[(test_df.c7525!=-1) & (test_df.c7525==test_df.Age)].shape[0]/test_df[test_df.c7525!=-1].shape[0]}')
print(f'Porcentagem classificadas corretamente das totais: {test_df[(test_df.c7525!=-1) & (test_df.c7525==test_df.Age)].shape[0]/test_df.shape[0]}')

/tmp/ipykernel_28074/319052794.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c7525'][index] = 0
/tmp/ipykernel_28074/319052794.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c7525'][index] = 2
/tmp/ipykernel_28074/319052794.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['c7525'][index] = 1


Instâncias totais: 9553
Instâncias classificadas: 1605
Porcentagem classificadas: 0.16801004919920443
Instâncias classificadas corretamente: 1044
Porcentagem classificadas corretamente: 0.6504672897196262
Porcentagem classificadas corretamente das totais: 0.10928504134826755
